# PySpark 101

In [75]:
# najprv ho nainstalujeme:
!pip install pyspark

In [76]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('Sensor Report')
    .getOrCreate()
)


## Reading Dataset

In [77]:
df = spark.read.csv(
    'datasets/sensors.csv.gz',
    header=True,
    inferSchema=True,
)
df

DataFrame[epoch: int, moteid: int, temperature: double, humidity: double, light: double, voltage: double, timestamp: timestamp, x: double, y: double]

In [78]:
schema = """
    epoch INT, moteid INT, temperature DOUBLE, humidity DOUBLE,
    light DOUBLE, voltage DOUBLE, timestamp TIMESTAMP, x DOUBLE, y DOUBLE
"""
df = spark.read.csv(
    'datasets/sensors.csv.gz',
    header=True,
    schema=schema
)

In [79]:
df.printSchema()

root
 |-- epoch: integer (nullable = true)
 |-- moteid: integer (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- light: double (nullable = true)
 |-- voltage: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- x: double (nullable = true)
 |-- y: double (nullable = true)



In [80]:
df

DataFrame[epoch: int, moteid: int, temperature: double, humidity: double, light: double, voltage: double, timestamp: timestamp, x: double, y: double]

In [81]:
df.show()

+-----+------+-----------+--------+-----+-------+-------------------+----+----+
|epoch|moteid|temperature|humidity|light|voltage|          timestamp|   x|   y|
+-----+------+-----------+--------+-----+-------+-------------------+----+----+
|    3|     1|      19.99|   37.09|45.08|    2.7|2004-02-28 00:59:16|21.5|23.0|
|   11|     1|       19.3|   38.46|45.08|   2.69|2004-02-28 01:03:16|21.5|23.0|
|   17|     1|      19.17|    38.8|45.08|   2.69|2004-02-28 01:06:16|21.5|23.0|
|   18|     1|      19.17|   38.84|45.08|    2.7|2004-02-28 01:06:46|21.5|23.0|
|   22|     1|      19.15|   38.94|45.08|   2.69|2004-02-28 01:08:45|21.5|23.0|
|   23|     1|      19.17|   38.87|45.08|   2.69|2004-02-28 01:09:22|21.5|23.0|
|   24|     1|      19.17|    38.8|45.08|   2.69|2004-02-28 01:09:46|21.5|23.0|
|   25|     1|      19.15|   38.84|45.08|    2.7|2004-02-28 01:10:16|21.5|23.0|
|   26|     1|      19.15|   38.87|45.08|   2.69|2004-02-28 01:10:46|21.5|23.0|
|   28|     1|      19.15|   38.94|45.08

In [82]:
df.show(5, truncate=False)

+-----+------+-----------+--------+-----+-------+-------------------+----+----+
|epoch|moteid|temperature|humidity|light|voltage|timestamp          |x   |y   |
+-----+------+-----------+--------+-----+-------+-------------------+----+----+
|3    |1     |19.99      |37.09   |45.08|2.7    |2004-02-28 00:59:16|21.5|23.0|
|11   |1     |19.3       |38.46   |45.08|2.69   |2004-02-28 01:03:16|21.5|23.0|
|17   |1     |19.17      |38.8    |45.08|2.69   |2004-02-28 01:06:16|21.5|23.0|
|18   |1     |19.17      |38.84   |45.08|2.7    |2004-02-28 01:06:46|21.5|23.0|
|22   |1     |19.15      |38.94   |45.08|2.69   |2004-02-28 01:08:45|21.5|23.0|
+-----+------+-----------+--------+-----+-------+-------------------+----+----+
only showing top 5 rows


In [83]:
df.count()

1868822

## Lazy Evaluation

In [84]:
df.filter( (df['temperature'] > 30) & (df['moteid'] == 1) ).explain()

== Physical Plan ==
*(1) Filter (((isnotnull(temperature#2677) AND isnotnull(moteid#2676)) AND (temperature#2677 > 30.0)) AND (moteid#2676 = 1))
+- FileScan csv [epoch#2675,moteid#2676,temperature#2677,humidity#2678,light#2679,voltage#2680,timestamp#2681,x#2682,y#2683] Batched: false, DataFilters: [isnotnull(temperature#2677), isnotnull(moteid#2676), (temperature#2677 > 30.0), (moteid#2676 = 1)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/mirek_binas/airflow/notebooks/datasets/sensors.csv.gz], PartitionFilters: [], PushedFilters: [IsNotNull(temperature), IsNotNull(moteid), GreaterThan(temperature,30.0), EqualTo(moteid,1)], ReadSchema: struct<epoch:int,moteid:int,temperature:double,humidity:double,light:double,voltage:double,timest...




In [85]:
df.filter( df['temperature'] > 30 ).filter( df['moteid'] == 1 ).explain()

== Physical Plan ==
*(1) Filter (((isnotnull(temperature#2677) AND isnotnull(moteid#2676)) AND (temperature#2677 > 30.0)) AND (moteid#2676 = 1))
+- FileScan csv [epoch#2675,moteid#2676,temperature#2677,humidity#2678,light#2679,voltage#2680,timestamp#2681,x#2682,y#2683] Batched: false, DataFilters: [isnotnull(temperature#2677), isnotnull(moteid#2676), (temperature#2677 > 30.0), (moteid#2676 = 1)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/mirek_binas/airflow/notebooks/datasets/sensors.csv.gz], PartitionFilters: [], PushedFilters: [IsNotNull(temperature), IsNotNull(moteid), GreaterThan(temperature,30.0), EqualTo(moteid,1)], ReadSchema: struct<epoch:int,moteid:int,temperature:double,humidity:double,light:double,voltage:double,timest...




## Column Selection

In [87]:
df.select('timestamp', 'moteid', 'temperature').show()

+-------------------+------+-----------+
|          timestamp|moteid|temperature|
+-------------------+------+-----------+
|2004-02-28 00:59:16|     1|      19.99|
|2004-02-28 01:03:16|     1|       19.3|
|2004-02-28 01:06:16|     1|      19.17|
|2004-02-28 01:06:46|     1|      19.17|
|2004-02-28 01:08:45|     1|      19.15|
|2004-02-28 01:09:22|     1|      19.17|
|2004-02-28 01:09:46|     1|      19.17|
|2004-02-28 01:10:16|     1|      19.15|
|2004-02-28 01:10:46|     1|      19.15|
|2004-02-28 01:11:46|     1|      19.15|
|2004-02-28 01:12:46|     1|      19.14|
|2004-02-28 01:14:16|     1|      19.12|
|2004-02-28 01:14:46|     1|      19.12|
|2004-02-28 01:15:16|     1|      19.11|
|2004-02-28 01:16:16|     1|      19.11|
|2004-02-28 01:16:46|     1|       19.1|
|2004-02-28 01:17:46|     1|       19.1|
|2004-02-28 01:18:16|     1|      19.09|
|2004-02-28 01:20:16|     1|      19.07|
|2004-02-28 01:20:46|     1|      19.07|
+-------------------+------+-----------+
only showing top

In [ ]:
(
    df
    .filter(df['temperature'] > 30)
    .filter(df['moteid'] == 1)
    .select('timestamp', 'moteid', 'temperature', 'humidity')
    .show()
)

+-------------------+------+-----------+--------+
|          timestamp|moteid|temperature|humidity|
+-------------------+------+-----------+--------+
|2004-03-23 01:24:28|     1|      30.07|   49.19|
|2004-03-23 01:26:02|     1|      30.02|   49.23|
|2004-03-23 01:27:28|     1|      30.08|   49.52|
|2004-03-23 01:28:29|     1|      30.24|   49.23|
|2004-03-23 01:29:28|     1|      30.34|   48.84|
|2004-03-23 01:29:58|     1|      30.18|   49.13|
|2004-03-23 01:32:01|     1|      30.19|   49.13|
|2004-03-23 01:32:33|     1|      30.16|   49.16|
|2004-03-23 01:34:58|     1|      30.44|   49.26|
|2004-03-23 01:36:29|     1|      30.35|   49.23|
|2004-03-23 01:36:59|     1|      30.44|   49.23|
|2004-03-23 01:37:47|     1|      30.42|   49.19|
|2004-03-23 01:38:02|     1|       30.3|   49.23|
|2004-03-23 01:39:09|     1|      30.37|   49.29|
|2004-03-23 01:40:05|     1|      30.49|   49.23|
|2004-03-23 01:40:40|     1|       30.6|   49.26|
|2004-03-23 01:41:06|     1|      30.55|   49.32|


In [88]:
(
    df
    .where('temperature > 30')
    .where('moteid == 1')
    .select('timestamp', 'moteid', 'temperature', 'humidity')
    .explain()
)

== Physical Plan ==
*(1) Project [timestamp#2681, moteid#2676, temperature#2677, humidity#2678]
+- *(1) Filter (((isnotnull(temperature#2677) AND isnotnull(moteid#2676)) AND (temperature#2677 > 30.0)) AND (moteid#2676 = 1))
   +- FileScan csv [moteid#2676,temperature#2677,humidity#2678,timestamp#2681] Batched: false, DataFilters: [isnotnull(temperature#2677), isnotnull(moteid#2676), (temperature#2677 > 30.0), (moteid#2676 = 1)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/mirek_binas/airflow/notebooks/datasets/sensors.csv.gz], PartitionFilters: [], PushedFilters: [IsNotNull(temperature), IsNotNull(moteid), GreaterThan(temperature,30.0), EqualTo(moteid,1)], ReadSchema: struct<moteid:int,temperature:double,humidity:double,timestamp:timestamp>




## Adding Columns

In [89]:
from pyspark.sql import functions as F

df = (
    df
        .withColumn('date', F.to_date('timestamp'))
        .withColumn('hour', F.hour('timestamp'))
        .withColumn('zone',
            F.when(F.col('x') < 14, 'east')        # if df['x'] < 14 : return 'east'
                .when(F.col('x') < 28, 'center')   # elif df['x'] < 28 : return 'center'
                .otherwise('west')                 # else: return 'west'
        )
)





## Descriptive Statistics

In [90]:
df.describe().show()

+-------+------------------+------------------+------------------+-----------------+-----------------+-------------------+-----------------+------------------+------------------+-------+
|summary|             epoch|            moteid|       temperature|         humidity|            light|            voltage|                x|                 y|              hour|   zone|
+-------+------------------+------------------+------------------+-----------------+-----------------+-------------------+-----------------+------------------+------------------+-------+
|  count|           1868822|           1868822|           1868822|          1868822|          1792768|            1868822|          1868822|           1868822|           1868822|1868822|
|   mean|29984.644200999344|28.871146101661903|22.110918359265874|39.53201970541372|409.3076149351448|   2.55862711376549|21.09768747371339|  18.7115546584961|11.459004656409224|   NULL|
| stddev|18525.936395283796|15.153111759979675| 3.227383235014166

## Aggregation Functions


In [ ]:
df.select(
    F.count('temperature').alias('count'),
    F.min('temperature').alias('min'),
    F.max('temperature').alias('max'),
    F.mean('temperature').alias('mean')
).show()

+-------+-----+-----+----+
|  count|  min|  max|mean|
+-------+-----+-----+----+
|1868822|10.19|34.99|22.0|
+-------+-----+-----+----+



In [110]:
import math

round(12.445)

12

In [105]:
math.floor(12.9345)

12

In [107]:
math.ceil(12.145)

13

In [ ]:
math.